# 👥 Notebook 02 — Collaborative filtering og ALS

**Metadata ser filmene. Men hva om vi ser på *menneskene* i stedet?**

Tusenvis av brukere har allerede fortalt oss hva de liker — gjennom handlingene sine.
Nå lar vi det kollektive mønsteret hjelpe Lea. Vi går fra en enkel item-item-metode
til den klassiske arbeidshesten **ALS matrisfaktorisering**.

## Fra nabolag til latente faktorer

### Item-item collaborative filtering

Vi slutter å lese filmomslaget og spør i stedet: *hvem andre så denne filmen,
og hva så de etterpå?* Det er lett å forstå og gir en god inngang til
collaborative filtering.

### Matrix factorization

I stedet for å sammenligne rå item-vektorer direkte, lærer vi latente faktorer:

$$R \approx U \cdot V^T$$

Her er $U$ brukerfaktorer og $V$ itemfaktorer. ALS er en effektiv måte å lære dette
på for store, sparsomme implisitte datasett.

## Oppsett

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from src.data import load_interactions, load_item_metadata, GENRE_COLS
from src.split import leave_one_out_split, build_sparse_matrix
from src.metrics import recall_at_k, ndcg_at_k, map_at_k
from implicit.als import AlternatingLeastSquares

interactions = load_interactions()
items = load_item_metadata()
train_df, test_df = leave_one_out_split(interactions)
n_users = interactions.user_id.max() + 1
n_items = interactions.item_id.max() + 1
train_matrix = build_sparse_matrix(train_df, n_users, n_items)
user_ids = test_df['user_id'].values
test_items = test_df['item_id'].values
K = 10
LEA_ID = 451

print(f'Matrise: {train_matrix.shape}, nnz={train_matrix.nnz:,}')

## 🏋️ Oppgave 2 — Inspiser item-likhet

In [ ]:
item_sim = cosine_similarity(train_matrix.T, dense_output=True)
np.fill_diagonal(item_sim, 0.0)
sample_items = items.sample(3, random_state=42)

for _, row in sample_items.iterrows():
    item_id = row['item_id']
    if item_id >= item_sim.shape[0]:
        continue
    neighbors = np.argsort(-item_sim[item_id])[:5]
    print(f'\n"{row["title"]}" ligner på:')
    for neighbor_id in neighbors:
        neighbor_row = items[items.item_id == neighbor_id]
        if len(neighbor_row) > 0:
            print(f'  {item_sim[item_id, neighbor_id]:.3f}  {neighbor_row.iloc[0]["title"]}')

> 💬 **Diskuter**
>
> 1. Ser naboskapet meningsfullt ut?
> 2. Hva fanger item-item-likheten som metadata alene ikke så godt viser?
> 3. Hvorfor er dette fortsatt begrenset?

## 🏋️ Oppgave 3 — Head-to-head: item-item vs ALS

Nå setter vi de to mot hverandre. Se spesielt på Recall og NDCG —
tar ALS et tydelig sprang?

In [ ]:
def recommend_item_item(item_sim, train_matrix, user_ids, k=10):
    recommendations = np.zeros((len(user_ids), k), dtype=np.int32)
    for row_index, user_id in enumerate(user_ids):
        user_vector = train_matrix[user_id]
        scores = item_sim @ user_vector.T
        scores = np.asarray(scores).flatten()
        scores[user_vector.indices] = -np.inf
        recommendations[row_index] = np.argsort(-scores)[:k]
    return recommendations

recs_item_item = recommend_item_item(item_sim, train_matrix, user_ids, k=K)

als = AlternatingLeastSquares(factors=64, regularization=0.01, iterations=15, random_state=42, use_gpu=False)
als.fit(train_matrix, show_progress=True)
recs_als = als.recommend(user_ids, train_matrix[user_ids], N=K, filter_already_liked_items=True)[0]

for name, recs in [('Item-item', recs_item_item), ('ALS', recs_als)]:
    recall_value = recall_at_k(recs, test_items, K)
    ndcg_value = ndcg_at_k(recs, test_items, K)
    map_value = map_at_k(recs, test_items, K)
    print(f'{name:<10} Recall@{K}={recall_value:.4f}  NDCG@{K}={ndcg_value:.4f}  MAP@{K}={map_value:.4f}')

In [ ]:
lea_idx = np.where(user_ids == LEA_ID)[0]
if len(lea_idx) > 0:
    for name, recs in [('Item-item', recs_item_item), ('ALS', recs_als)]:
        lea_recs = recs[lea_idx[0]]
        titles = items.set_index('item_id').loc[lea_recs, 'title'].values
        print(f'\n{name}:')
        for rank, title in enumerate(titles, 1):
            print(f'  {rank:>2}. {title}')

ALS tar et tydelig sprang. Se på Leas lister — ser du forskjellen?

> 💬 **Diskuter**
>
> 1. Hvorfor vinner ALS ofte over item-item?
> 2. Hva er forskjellen på en nabolagsmetode og latente faktorer?
> 3. Ser ALS mer ut som et realistisk produksjonsvalg enn item-item? Hvorfor?

### 🔄 Refleksjon — sjekk prediksjonen din

Gå tilbake til gjetningene du skrev ned i notebook 00.

> 1. Hadde du rett om hva Lea ville like?
> 2. Stemte gjetningen din om popularitetslisten?
> 3. Hva har overrasket deg mest så langt?

---

> *Marte:* «Bra. Nå ser det ut som Lea kan få bedre anbefalinger.
> Men fungerer dette for *alle* — eller bare for mainstream-brukerne?
> Og hva skjer når vi må blande signaler og ta fairness på alvor?»

**Neste steg** → `03_hybrid_systems.ipynb`

## Valgfri appendix — faktorsveip og tolkning

Denne delen er nyttig hvis dere vil grave dypere i hvordan ALS oppfører seg,
men den er ikke nødvendig for kjerneflyten.

In [ ]:
import matplotlib.pyplot as plt

factor_values = [16, 32, 64, 128, 256]
recall_per_factor = []
for factor_count in factor_values:
    model = AlternatingLeastSquares(factors=factor_count, regularization=0.01, iterations=15, random_state=42, use_gpu=False)
    model.fit(train_matrix, show_progress=False)
    recs = model.recommend(user_ids, train_matrix[user_ids], N=K, filter_already_liked_items=True)[0]
    recall_per_factor.append(recall_at_k(recs, test_items, K))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(factor_values, recall_per_factor, 'o-', linewidth=2)
ax.set_xlabel('Antall faktorer')
ax.set_ylabel(f'Recall@{K}')
ax.set_title('ALS: Recall vs antall faktorer')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cosine_sim

user_emb = als.user_factors
lea_emb = user_emb[LEA_ID]
sims = cosine_sim(lea_emb.reshape(1, -1), user_emb)[0]
sims[LEA_ID] = -1
top_neighbors = np.argsort(-sims)[:5]

print('Leas nærmeste naboer i ALS-rommet:')
for neighbor_user_id in top_neighbors:
    neighbor_items = interactions[interactions.user_id == neighbor_user_id].merge(items, on='item_id')
    top_genres = neighbor_items[GENRE_COLS].sum().nlargest(3).index.tolist()
    genre_label = ', '.join(top_genres)
    print(f'  Bruker {neighbor_user_id} (sim={sims[neighbor_user_id]:.3f}, topp-sjangre: {genre_label})')